In [1]:
import sys
from pathlib import Path
sys.path.append(f"{Path().absolute().parent}")

In [2]:
import pandas as pd
from notebooks.radp_library import (preprocess_ue_data)
from apps.mobility_robustness_optimization.simple_mro import SimpleMRO
from apps.mobility_robustness_optimization.mro_rl import ReinforcedMRO

In [3]:
simple_ue = pd.read_csv('E:/Repositories/maveric/notebooks/data/sim_data/UE_data_20UE_100ticks.csv')
topology = pd.read_csv('E:/Repositories/maveric/notebooks/data/sim_data/topology.csv')
topology.loc[topology["cell_id"] == "cell_1", "cell_lat"] = -90
topology.loc[topology["cell_id"] == "cell_2", "cell_lat"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lat"] = 90

topology.loc[topology["cell_id"] == "cell_1", "cell_lon"] = -180
topology.loc[topology["cell_id"] == "cell_2", "cell_lon"] = 0
topology.loc[topology["cell_id"] == "cell_3", "cell_lon"] = 180

topology.loc[topology["cell_id"] == "cell_1", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_2", "cell_carrier_freq_mhz"] = 2100
topology.loc[topology["cell_id"] == "cell_3", "cell_carrier_freq_mhz"] = 2100

simple_ue.drop(columns=['mock_ue_id', 'tick'], inplace=True)
simple_ue

,longitude,latitude
0,-22.625309,59.806764
1,119.764151,54.857584
2,72.095437,-20.253892
3,-67.548009,-38.100941
4,59.867089,-83.103930
...,...,...
1995,45.564260,42.609846
1996,132.457280,17.241235
1997,-101.217659,72.295988
1998,-16.480045,-26.656397


In [4]:
topology

,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz
0,-90.0,-180.0,cell_1,0,2100
1,0.0,0.0,cell_2,120,2100
2,90.0,180.0,cell_3,240,2100


In [5]:
params = {
    "ue_tracks_generation": {
            "params": {
                "simulation_duration": 3600,
                "simulation_time_interval_seconds": 0.01,
                "num_ticks": 100,
                "num_batches": 1,
                "ue_class_distribution": {
                    "stationary": {
                        "count": 5,
                        "velocity": 0,
                        "velocity_variance": 1
                    },
                    "pedestrian": {
                        "count": 5,
                        "velocity": 2,
                        "velocity_variance": 1
                    },
                    "cyclist": {
                        "count": 5,
                        "velocity": 5,
                        "velocity_variance": 1
                    },
                    "car": {
                        "count": 5,
                        "velocity": 20,
                        "velocity_variance": 1
                    }
                },
                "lat_lon_boundaries": {
                    "min_lat": -90,
                    "max_lat": 90,
                    "min_lon": -180,
                    "max_lon": 180
                },
                "gauss_markov_params": {
                    "alpha": 0.5,
                    "variance": 0.8,
                    "rng_seed": 42,
                    "lon_x_dims": 100,
                    "lon_y_dims": 100,
                    "// TODO": "Account for supporting the user choosing the anchor_loc and cov_around_anchor.",
                    "// Current implementation": "the UE Tracks generator will not be using these values.",
                    "// anchor_loc": {},
                    "// cov_around_anchor": {}
            }
        }
    }
}

__Start__

In [6]:
input_data = preprocess_ue_data(simple_ue, topology)
input_data

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm
0,-22.625309,59.806764,-90.0,-180.0,1,0,2100,16.629500,-100.311970
1,-22.625309,59.806764,0.0,0.0,2,120,2100,15.752768,-99.841523
2,-22.625309,59.806764,90.0,180.0,3,240,2100,15.027772,-99.432278
3,119.764151,54.857584,-90.0,-180.0,1,0,2100,16.595905,-100.294405
4,119.764151,54.857584,0.0,0.0,2,120,2100,16.289273,-100.132420
...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,0.0,0.0,2,120,2100,15.054748,-99.447856
5996,-16.480045,-26.656397,90.0,180.0,3,240,2100,16.379387,-100.180339
5997,34.222834,63.970820,-90.0,-180.0,1,0,2100,16.656917,-100.326278
5998,34.222834,63.970820,0.0,0.0,2,120,2100,15.850264,-99.895116


### Simple MRO

__init__

In [7]:
mro = SimpleMRO(mobility_model_params = params, topology = topology)

__update__

In [8]:
mro.train_or_update_rf_twin(input_data)

No Bayesian Digital Twins available for update. Training from scratch.


[2025-05-10 14:08:56,922] INFO:  Iter 1/100 - Loss: 0.784 (delta=inf)
[2025-05-10 14:08:57,069] INFO:  Iter 2/100 - Loss: 0.765 (delta=-0.019080)
[2025-05-10 14:08:57,228] INFO:  Iter 3/100 - Loss: 0.745 (delta=-0.019101)
[2025-05-10 14:08:57,371] INFO:  Iter 4/100 - Loss: 0.726 (delta=-0.019147)
[2025-05-10 14:08:57,518] INFO:  Iter 5/100 - Loss: 0.707 (delta=-0.019220)
[2025-05-10 14:08:57,655] INFO:  Iter 6/100 - Loss: 0.688 (delta=-0.019358)
[2025-05-10 14:08:57,787] INFO:  Iter 7/100 - Loss: 0.668 (delta=-0.019525)
[2025-05-10 14:08:57,913] INFO:  Iter 8/100 - Loss: 0.648 (delta=-0.019719)
[2025-05-10 14:08:58,045] INFO:  Iter 9/100 - Loss: 0.629 (delta=-0.019950)
[2025-05-10 14:08:58,173] INFO:  Iter 10/100 - Loss: 0.608 (delta=-0.020138)
[2025-05-10 14:08:58,299] INFO:  Iter 11/100 - Loss: 0.588 (delta=-0.020339)
[2025-05-10 14:08:58,417] INFO:  Iter 12/100 - Loss: 0.567 (delta=-0.020539)
[2025-05-10 14:08:58,545] INFO:  Iter 13/100 - Loss: 0.547 (delta=-0.020725)
[2025-05-10 14

In [9]:
mro.bayesian_digital_twins

{'cell_1': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x1d8490a2cd0>,
 'cell_2': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x1d8255d2700>,
 'cell_3': <radp.digital_twin.rf.bayesian.bayesian_engine.BayesianDigitalTwin at 0x1d849135070>}

__solve__

In [10]:
mro.solve()

e:\Repositories\maveric\.venv\lib\site-packages\gpytorch\distributions\multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
e:\Repositories\maveric\.venv\lib\site-packages\gpytorch\distributions\multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
e:\Repositories\maveric\.venv\lib\site-packages\gpytorch\distributions\multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-06.
  warnings.warn(
e:\Repositories\maveric\.venv\lib\site-packages\gpytorch\distributions\multivariate_normal.py:319: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up

Epoch  Hyst           TTT    MRO Metric  
-----------------------------------------
0      3.1582717095   62     99.350000   
1      3.8788105877   14     99.850000   
2      3.1172331095   10     100.000000  
3      4.0460215430   76     99.300000   
4      3.1482105634   17     99.750000   
5      2.7659850041   63     99.350000   
6      4.0140340921   67     99.350000   
7      1.6198089543   84     99.250000   
8      2.7246807445   31     99.600000   
9      0.4396126301   2      99.950000   
10     3.5901407516   11     100.000000  
11     0.6142438519   42     99.550000   
12     1.2851398145   5      100.000000  
13     2.8878472879   78     99.300000   
14     0.6839633563   82     99.300000   
15     3.1045648162   35     99.550000   
16     1.1314945183   86     99.150000   
17     2.1038873008   60     99.350000   
18     4.0894394660   3      100.000000  
19     3.2053840459   4      100.000000  
20     0.1492972897   25     99.650000   
21     1.0219787213   32     99.55

(3.1172331095138524, 10)

In [11]:
mro.solve()

Epoch  Hyst           TTT    MRO Metric  
-----------------------------------------
0      1.7639327835   39     99.550000   
1      2.6361970214   30     99.650000   
2      1.9588965632   4      100.000000  
3      0.9877411034   27     99.650000   
4      2.4459890405   18     99.700000   
5      2.8002814540   38     99.550000   
6      1.1594129407   78     99.300000   
7      3.9577477030   90     99.150000   
8      3.3210715569   39     99.550000   
9      3.9663343374   95     99.150000   
10     1.4497641215   84     99.250000   
11     0.4477494902   87     99.150000   
12     1.4644293616   49     99.450000   
13     2.4410799000   12     99.950000   
14     1.7552598574   58     99.350000   
15     2.4278209601   20     99.650000   
16     1.9578000841   54     99.400000   
17     0.4074081844   98     99.150000   
18     1.9838415046   77     99.300000   
19     2.9610700859   87     99.150000   
20     0.0073801038   88     99.150000   
21     2.5617725796   65     99.35

(1.95889656321369, 4)

In [12]:
mro.train_or_update_rf_twin(input_data)

ValueError: The input DataFrame must contain the following columns: {'cell_az_deg', 'cell_rxpwr_dbm', 'longitude', 'cell_lat', 'cell_lon', 'cell_carrier_freq_mhz', 'cell_id', 'latitude'}


### RL MRO

In [13]:
rl_mro = ReinforcedMRO(mobility_model_params = params, topology = topology)

In [17]:
input_data = preprocess_ue_data(simple_ue, topology)

In [19]:
rl_mro.train_or_update_rf_twin(input_data)

[2025-05-10 14:36:03,564] INFO:  Iter 1/100 - Loss: 0.784 (delta=inf)


No Bayesian Digital Twins available for update. Training from scratch.


[2025-05-10 14:36:03,728] INFO:  Iter 2/100 - Loss: 0.765 (delta=-0.019080)
[2025-05-10 14:36:03,870] INFO:  Iter 3/100 - Loss: 0.745 (delta=-0.019100)
[2025-05-10 14:36:04,012] INFO:  Iter 4/100 - Loss: 0.726 (delta=-0.019144)
[2025-05-10 14:36:04,153] INFO:  Iter 5/100 - Loss: 0.707 (delta=-0.019228)
[2025-05-10 14:36:04,296] INFO:  Iter 6/100 - Loss: 0.688 (delta=-0.019357)
[2025-05-10 14:36:04,437] INFO:  Iter 7/100 - Loss: 0.668 (delta=-0.019525)
[2025-05-10 14:36:04,573] INFO:  Iter 8/100 - Loss: 0.648 (delta=-0.019734)
[2025-05-10 14:36:04,709] INFO:  Iter 9/100 - Loss: 0.629 (delta=-0.019926)
[2025-05-10 14:36:04,864] INFO:  Iter 10/100 - Loss: 0.608 (delta=-0.020139)
[2025-05-10 14:36:05,020] INFO:  Iter 11/100 - Loss: 0.588 (delta=-0.020334)
[2025-05-10 14:36:05,160] INFO:  Iter 12/100 - Loss: 0.567 (delta=-0.020559)
[2025-05-10 14:36:05,301] INFO:  Iter 13/100 - Loss: 0.547 (delta=-0.020715)
[2025-05-10 14:36:05,452] INFO:  Iter 14/100 - Loss: 0.526 (delta=-0.020919)
[2025-0

In [18]:
input_data

,longitude,latitude,cell_lat,cell_lon,cell_id,cell_az_deg,cell_carrier_freq_mhz,log_distance,cell_rxpwr_dbm
0,-22.625309,59.806764,-90.0,-180.0,1,0,2100,16.629500,-100.311970
1,-22.625309,59.806764,0.0,0.0,2,120,2100,15.752768,-99.841523
2,-22.625309,59.806764,90.0,180.0,3,240,2100,15.027772,-99.432278
3,119.764151,54.857584,-90.0,-180.0,1,0,2100,16.595905,-100.294405
4,119.764151,54.857584,0.0,0.0,2,120,2100,16.289273,-100.132420
...,...,...,...,...,...,...,...,...,...
5995,-16.480045,-26.656397,0.0,0.0,2,120,2100,15.054748,-99.447856
5996,-16.480045,-26.656397,90.0,180.0,3,240,2100,16.379387,-100.180339
5997,34.222834,63.970820,-90.0,-180.0,1,0,2100,16.656917,-100.326278
5998,34.222834,63.970820,0.0,0.0,2,120,2100,15.850264,-99.895116


In [ ]:
rl_mro.solve()

e:\Repositories\maveric\.venv\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


Using cpu device
Episode: 1, Timestep: 1, Hyst: 0.359125, TTT: 2, Reward: 99.950000, Done: False
Episode: 1, Timestep: 2, Hyst: 1.058560, TTT: 2, Reward: 100.000000, Done: False
Episode: 1, Timestep: 3, Hyst: 1.066607, TTT: 2, Reward: 100.000000, Done: False
Episode: 1, Timestep: 4, Hyst: 0.000000, TTT: 2, Reward: 99.450000, Done: False
Episode: 1, Timestep: 5, Hyst: 0.000000, TTT: 2, Reward: 99.450000, Done: False
